In [1]:
pip install mediapipe --user

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install --upgrade protobuf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import urllib.request

url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
urllib.request.urlretrieve(url, "face_landmarker.task")

print("Model downloaded ✅")

Model downloaded ✅


In [4]:
import cv2
import mediapipe as mp
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Load model
base_options = python.BaseOptions(model_asset_path="face_landmarker.task")

options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO
)

detector = vision.FaceLandmarker.create_from_options(options)

# Load glasses image
glasses = cv2.imread("sunglass2.png", cv2.IMREAD_UNCHANGED)

cap = cv2.VideoCapture(0)

timestamp = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    result = detector.detect_for_video(mp_image, timestamp)
    timestamp += 1

    if result.face_landmarks:
        for face_landmarks in result.face_landmarks:

            h, w, _ = frame.shape

            # LEFT & RIGHT EYE (approx indices)
            left_eye = face_landmarks[33]
            right_eye = face_landmarks[263]

            x1, y1 = int(left_eye.x * w), int(left_eye.y * h)
            x2, y2 = int(right_eye.x * w), int(right_eye.y * h)

            scale = 1.8   # 🔺 increase this (try 1.5 → 2.5)

            glasses_width = int(np.hypot(x2 - x1, y2 - y1) * scale)
            glasses_height = int(glasses_width * 0.5)
            if glasses is None:
                continue

            resized = cv2.resize(glasses, (glasses_width, glasses_height))

            x_center = (x1 + x2) // 2
            y_center = (y1 + y2) // 2

            x_offset = int(x_center - glasses_width / 2)
            y_offset = int(y_center - glasses_height / 2 -10)

            # boundary check
            if y_offset < 0 or x_offset < 0 or \
               y_offset + glasses_height > h or \
               x_offset + glasses_width > w:
                continue

            overlay_img = resized[:, :, :3]
            mask = resized[:, :, 3]
            mask_inv = cv2.bitwise_not(mask)

            roi = frame[y_offset:y_offset+glasses_height, x_offset:x_offset+glasses_width]

            bg = cv2.bitwise_and(roi, roi, mask=mask_inv)
            fg = cv2.bitwise_and(overlay_img, overlay_img, mask=mask)

            frame[y_offset:y_offset+glasses_height, x_offset:x_offset+glasses_width] = cv2.add(bg, fg)

    cv2.imshow("Face Filter 😎", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()